# 인용 참고문헌 arXiv 메타데이터: OAI-PMH 대량 수집

참조 논문당 한 행을 만들고 `cit_arxiv_id`에 인용 원본 논문 ID 목록을 저장합니다.

`export.arxiv.org/api/query`(Atom API)는 **캐시에 없는 요청을 전부 `429 Rate exceeded`로 거절**합니다. 고유한 `id_list` 조합은 언제나 캐시 미스이므로 요청 간격이나 배치 크기를 아무리 조정해도 통과하지 못합니다(User-Agent·호스트·스킴 변경도 모두 429 확인).

대신 OAI-PMH `ListRecords`로 arXiv 전체를 페이지 단위(1,300건/페이지)로 훑으면서 필요한 참조 논문만 골라냅니다. 전체 수확에 3~4시간 정도 걸리지만, 중단해도 `resumptionToken`이 저장되어 다시 실행하면 이어서 진행합니다.

In [1]:
import json
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists() and (p / 'data' / 'citations_ai').is_dir()), None)
if ROOT is None: raise FileNotFoundError('프로젝트 루트 또는 notebooks/에서 실행하세요.')
INPUT_FILES = [ROOT / 'data' / 'citations_ai' / 'arxiv_citations_part1.jsonl', ROOT / 'data' / 'citations_ai' / 'arxiv_citations_part2.jsonl']
if missing := [p for p in INPUT_FILES if not p.exists()]: raise FileNotFoundError(missing)
OUTPUT_DIR = ROOT / 'data' / 'ai_references' / 'arxiv_ai_references_oai_bulk'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FILE_PREFIX, CHUNK_SIZE = 'arxiv_ai_references_oai_bulk', 5_000
CACHE_PATH, STATE_PATH = OUTPUT_DIR / f'{FILE_PREFIX}_metadata_cache.jsonl', OUTPUT_DIR / f'{FILE_PREFIX}_state.json'
print(f'입력: {len(INPUT_FILES)}개 / 출력: {OUTPUT_DIR}')

입력: 2개 / 출력: c:\Users\oh\Desktop\arxiv_graph_RAG\data\ai_references\arxiv_ai_references_oai_bulk


In [2]:
import json
import re
import xml.etree.ElementTree as ET
from pathlib import Path
ARXIV_ID_RE = re.compile(r'^(?:arXiv:)?(?P<id>(?:\d{4}\.\d{4,5}|[A-Za-z-]+(?:\.[A-Za-z-]+)?/\d{7}))(?:v\d+)?$')
OAI_NS = {'oai': 'http://www.openarchives.org/OAI/2.0/', 'arxiv': 'http://arxiv.org/OAI/arXiv/'}
OUTPUT_FIELDS = ('id','title','abstract','authors','categories','primary_category','published','updated','doi','pdf_url','source')
def normalize_arxiv_id(value):
    if not isinstance(value, str): return None
    value = value.strip().removeprefix('https://arxiv.org/abs/').removeprefix('http://arxiv.org/abs/')
    match = ARXIV_ID_RE.fullmatch(value)
    return match.group('id') if match else None
def batched(items, size):
    for start in range(0, len(items), size): yield items[start:start + size]
def collect_references(files):
    collected, by_id, skipped = [], {}, 0
    for path in files:
        with Path(path).open(encoding='utf-8-sig') as handle:
            for line_number, line in enumerate(handle, 1):
                if not line.strip(): continue
                try: row = json.loads(line)
                except json.JSONDecodeError as error: raise ValueError(f'{path.name}:{line_number} JSON 파싱 실패') from error
                cit_id = normalize_arxiv_id(row.get('arxiv_id') if isinstance(row, dict) else None)
                if not cit_id: raise ValueError(f'{path.name}:{line_number} 원본 arxiv_id 오류')
                items = row.get('references') or []
                if not isinstance(items, list): raise ValueError(f'{path.name}:{line_number} references는 목록이어야 합니다.')
                for item in items:
                    arxiv_id = normalize_arxiv_id(item.get('arxiv_id') if isinstance(item, dict) else None)
                    if not arxiv_id: skipped += 1; continue
                    record = by_id.get(arxiv_id)
                    if record is None:
                        record = {'arxiv_id': arxiv_id, 'cit_arxiv_id': []}; by_id[arxiv_id] = record; collected.append(record)
                    if cit_id not in record['cit_arxiv_id']: record['cit_arxiv_id'].append(cit_id)
    return collected, skipped
def _text(node, path):
    child = node.find(path, OAI_NS); return child.text.strip() if child is not None and child.text else None
def announced_year_month(arxiv_id):
    digits = arxiv_id.split('/')[1] if '/' in arxiv_id else arxiv_id[:4]
    year = int(digits[:2])
    return f'{1900 + year if year >= 91 else 2000 + year}-{digits[2:4]}'
def resolve_published(arxiv_id, created):
    # OAI의 <created>는 개정판이 있으면 최신 버전 날짜가 되어 최초 공개일과 어긋난다.
    # arXiv ID에 인코딩된 연월이 최초 공개 시점이므로, 어긋나면 ID 쪽을 신뢰한다.
    year_month = announced_year_month(arxiv_id)
    return f'{created}T00:00:00Z' if created and created[:7] == year_month else f'{year_month}-01T00:00:00Z'
def parse_oai_record(record):
    header = record.find('oai:header', OAI_NS)
    if header is None or header.attrib.get('status') == 'deleted': return None
    meta = record.find('oai:metadata/arxiv:arXiv', OAI_NS)
    if meta is None: return None
    arxiv_id = normalize_arxiv_id(_text(meta, 'arxiv:id'))
    if not arxiv_id: return None
    authors = [name for author in meta.findall('arxiv:authors/arxiv:author', OAI_NS)
               if (name := f"{_text(author, 'arxiv:forenames') or ''} {_text(author, 'arxiv:keyname') or ''}".strip())]
    categories = (_text(meta, 'arxiv:categories') or '').split()
    created = _text(meta, 'arxiv:created')
    updated = _text(meta, 'arxiv:updated') or created
    return arxiv_id, {'id': f'https://arxiv.org/abs/{arxiv_id}', 'title': ' '.join((_text(meta, 'arxiv:title') or '').split()), 'abstract': ' '.join((_text(meta, 'arxiv:abstract') or '').split()), 'authors': authors, 'categories': categories, 'primary_category': categories[0] if categories else None, 'published': resolve_published(arxiv_id, created), 'updated': f'{updated}T00:00:00Z' if updated else None, 'doi': _text(meta, 'arxiv:doi'), 'pdf_url': f'https://arxiv.org/pdf/{arxiv_id}', 'source': 'arxiv'}
def make_output_record(reference, metadata, found):
    record = {'arxiv_id': reference['arxiv_id'], 'cit_arxiv_id': reference['cit_arxiv_id'], 'found': found}
    record.update({field: (metadata or {}).get(field) for field in OUTPUT_FIELDS} if metadata else {'id':None,'title':None,'abstract':None,'authors':[],'categories':[],'primary_category':None,'published':None,'updated':None,'doi':None,'pdf_url':None,'source':'arxiv'})
    return record

In [3]:
import http.client, ssl, time, urllib.error, urllib.parse, urllib.request
import xml.etree.ElementTree as ET
import truststore
OAI_URL, REQUEST_INTERVAL_SECONDS, MAX_RETRIES, MAX_DELAY_SECONDS = 'https://export.arxiv.org/oai2', 3.0, 8, 600
RETRYABLE_HTTP_CODES = {429, 500, 502, 503, 504}
RETRYABLE_NETWORK_ERRORS = (urllib.error.URLError, TimeoutError, ConnectionError, http.client.HTTPException)
_last_request_at = 0.0
def _retry_delay(error, attempt):
    retry_after = getattr(error, 'headers', None) and error.headers.get('Retry-After')
    if retry_after:
        try: return min(max(float(retry_after), REQUEST_INTERVAL_SECONDS), MAX_DELAY_SECONDS)
        except ValueError: pass
    return min(10 * 2 ** attempt, MAX_DELAY_SECONDS)
def oai_request(params):
    # arXiv OAI-PMH는 과부하 시 503 + Retry-After로 흐름을 제어하므로 헤더 값을 그대로 따른다
    global _last_request_at
    request = urllib.request.Request(f'{OAI_URL}?{urllib.parse.urlencode(params)}', headers={'User-Agent':'arxiv-citation-reference-harvester/1.0'})
    for attempt in range(MAX_RETRIES + 1):
        wait = REQUEST_INTERVAL_SECONDS - (time.monotonic() - _last_request_at)
        if wait > 0: time.sleep(wait)
        try:
            _last_request_at = time.monotonic()
            with urllib.request.urlopen(request, timeout=180, context=truststore.SSLContext(ssl.PROTOCOL_TLS_CLIENT)) as response: return ET.fromstring(response.read())
        except urllib.error.HTTPError as error:
            if error.code not in RETRYABLE_HTTP_CODES or attempt >= MAX_RETRIES:
                raise RuntimeError(f'OAI-PMH HTTP {error.code}: {error.read()[:200]!r}') from error
            delay = _retry_delay(error, attempt)
            print(f'HTTP {error.code}: {delay:.0f}초 후 재시도 ({attempt + 1}/{MAX_RETRIES})')
            time.sleep(delay)
        except RETRYABLE_NETWORK_ERRORS as error:
            if attempt >= MAX_RETRIES: raise
            delay = min(10 * 2 ** attempt, MAX_DELAY_SECONDS)
            print(f'네트워크 오류 {error!r}: {delay:.0f}초 후 재시도 ({attempt + 1}/{MAX_RETRIES})')
            time.sleep(delay)
    raise RuntimeError('재시도 횟수 초과')

In [4]:
import os, tempfile
from datetime import datetime, timezone
def atomic_write_jsonl(path, rows):
    with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', newline='\n', dir=path.parent, delete=False) as handle:
        temp = Path(handle.name)
        for row in rows: handle.write(json.dumps(row, ensure_ascii=False) + '\n')
    os.replace(temp, path)
def append_cache_rows(rows):
    # 페이지마다 캐시 전체를 다시 쓰면 O(n^2)이 되므로 새로 찾은 것만 덧붙인다
    with CACHE_PATH.open('a', encoding='utf-8', newline='\n') as handle:
        for row in rows: handle.write(json.dumps(row, ensure_ascii=False) + '\n')
def load_cache():
    if not CACHE_PATH.exists(): return {}
    with CACHE_PATH.open(encoding='utf-8') as handle: return {row['arxiv_id']: {'found':row['found'], 'metadata':row['metadata']} for row in map(json.loads, filter(str.strip, handle))}
def save_state(**fields):
    STATE_PATH.write_text(json.dumps({**fields, 'updated_at': datetime.now(timezone.utc).isoformat()}, ensure_ascii=False), encoding='utf-8')

references, skipped = collect_references(INPUT_FILES)
targets = {r['arxiv_id'] for r in references}
cache = load_cache()
state = json.loads(STATE_PATH.read_text(encoding='utf-8')) if STATE_PATH.exists() else {}
print(f'고유 참조: {len(references):,}; 캐시됨: {len(cache):,}; ID 없음: {skipped:,}')

if not state.get('harvest_complete'):
    token = state.get('resumption_token')
    params = {'verb':'ListRecords', 'resumptionToken':token} if token else {'verb':'ListRecords', 'metadataPrefix':'arXiv'}
    pages, scanned, started = state.get('pages', 0), state.get('scanned', 0), time.monotonic()
    while True:
        root = oai_request(params)
        error = root.find('oai:error', OAI_NS)
        if error is not None:
            if error.attrib.get('code') != 'badResumptionToken': raise RuntimeError(f"OAI-PMH {error.attrib.get('code')}: {error.text}")
            print('resumptionToken이 만료되어 처음부터 다시 수확합니다.')
            params, pages, scanned = {'verb':'ListRecords', 'metadataPrefix':'arXiv'}, 0, 0
            continue
        new_rows = []
        for record in root.findall('.//oai:record', OAI_NS):
            scanned += 1
            parsed = parse_oai_record(record)
            if parsed is None: continue
            arxiv_id, metadata = parsed
            if arxiv_id in targets and arxiv_id not in cache:
                cache[arxiv_id] = {'found': True, 'metadata': metadata}
                new_rows.append({'arxiv_id': arxiv_id, 'found': True, 'metadata': metadata})
        if new_rows: append_cache_rows(new_rows)
        node = root.find('.//oai:resumptionToken', OAI_NS)
        token = node.text if node is not None and node.text else None
        pages += 1
        save_state(resumption_token=token, harvest_complete=token is None, pages=pages, scanned=scanned, cached_count=len(cache))
        if pages % 10 == 0 or token is None:
            print(f'{pages:,}페이지 / {scanned:,}건 스캔 / 대상 확보 {len(cache):,}/{len(targets):,} / 경과 {(time.monotonic() - started) / 60:.0f}분')
        if token is None:
            print('수확 완료')
            break
        params = {'verb':'ListRecords', 'resumptionToken':token}

if missing := [t for t in sorted(targets) if t not in cache]:
    append_cache_rows([{'arxiv_id': t, 'found': False, 'metadata': None} for t in missing])
    for t in missing: cache[t] = {'found': False, 'metadata': None}
    print(f'arXiv에서 찾지 못한 참조: {len(missing):,}건')

records = [make_output_record(r, cache[r['arxiv_id']]['metadata'], cache[r['arxiv_id']]['found']) for r in references]
for index, rows in enumerate(batched(records, CHUNK_SIZE), 1): atomic_write_jsonl(OUTPUT_DIR / f'{FILE_PREFIX}_part{index}.jsonl', rows)
print(f'저장 완료: {len(records):,}행')

고유 참조: 31,913; 캐시됨: 0; ID 없음: 60,549
10페이지 / 13,000건 스캔 / 대상 확보 2/31,913 / 경과 1분
20페이지 / 26,000건 스캔 / 대상 확보 4/31,913 / 경과 2분
30페이지 / 39,000건 스캔 / 대상 확보 35/31,913 / 경과 3분
40페이지 / 52,000건 스캔 / 대상 확보 36/31,913 / 경과 5분
50페이지 / 65,000건 스캔 / 대상 확보 38/31,913 / 경과 6분
60페이지 / 78,000건 스캔 / 대상 확보 42/31,913 / 경과 7분
70페이지 / 91,000건 스캔 / 대상 확보 46/31,913 / 경과 9분
80페이지 / 104,000건 스캔 / 대상 확보 52/31,913 / 경과 10분
90페이지 / 117,000건 스캔 / 대상 확보 58/31,913 / 경과 11분
100페이지 / 130,000건 스캔 / 대상 확보 63/31,913 / 경과 12분
110페이지 / 143,000건 스캔 / 대상 확보 76/31,913 / 경과 13분
120페이지 / 156,000건 스캔 / 대상 확보 87/31,913 / 경과 14분
130페이지 / 169,000건 스캔 / 대상 확보 90/31,913 / 경과 15분
140페이지 / 182,000건 스캔 / 대상 확보 93/31,913 / 경과 16분
150페이지 / 195,000건 스캔 / 대상 확보 93/31,913 / 경과 18분
160페이지 / 208,000건 스캔 / 대상 확보 95/31,913 / 경과 19분
170페이지 / 221,000건 스캔 / 대상 확보 100/31,913 / 경과 20분
180페이지 / 234,000건 스캔 / 대상 확보 105/31,913 / 경과 21분
190페이지 / 247,000건 스캔 / 대상 확보 113/31,913 / 경과 22분
200페이지 / 260,000건 스캔 / 대상 확보 119/31,913 / 경과 23분
210페이지 / 273,000건 스캔 / 대

KeyboardInterrupt: 

In [ ]:
saved = []
for path in sorted(OUTPUT_DIR.glob(f'{FILE_PREFIX}_part*.jsonl')):
    with path.open(encoding='utf-8') as handle: saved.extend(json.loads(line) for line in handle if line.strip())
assert len(saved) == len(references) == len({row['arxiv_id'] for row in saved})
assert {row['arxiv_id']:row['cit_arxiv_id'] for row in saved} == {row['arxiv_id']:row['cit_arxiv_id'] for row in references}
assert all(isinstance(row['cit_arxiv_id'], list) and row['cit_arxiv_id'] for row in saved)
assert all(row['title'] and row['abstract'] for row in saved if row['found'])
found_count = sum(row['found'] for row in saved)
print(f'검증 완료: 참조 논문 {len(saved):,}행, 인용 관계 {sum(len(row["cit_arxiv_id"]) for row in saved):,}건')
print(f'메타데이터 확보: {found_count:,}건 / 미확보: {len(saved) - found_count:,}건')
display(saved[:3])